# Financial Q&A - QLoRA Fine-Tuning on Qwen2.5-1.5B-Instruct

Fine-tune a small open-source language model for Financial Question Answering using QLoRA and demonstrate measurable improvement.

**Runtime**: Go to `Runtime > Change runtime type > T4 GPU`

## Step 0: Install Dependencies

In [ ]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes evaluate rouge-score scikit-learn pandas tqdm

In [ ]:
import json
import re
import random
from pathlib import Path

import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
from rouge_score import rouge_scorer
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Data Preparation

Load the **finance-alpaca** dataset from HuggingFace, clean it, and format into chat template.

In [ ]:
# --- Config ---
SAMPLE_SIZE = 1000
EVAL_RATIO = 0.1
SEED = 42
random.seed(SEED)

In [ ]:
# --- Load dataset ---
print("Loading finance-alpaca dataset...")
dataset = load_dataset("gbharti/finance-alpaca", split="train")
print(f"Raw dataset size: {len(dataset)}")

In [ ]:
# --- Clean and filter ---
def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()

def is_valid(sample):
    instruction = clean_text(sample.get("instruction", ""))
    output = clean_text(sample.get("output", ""))
    return len(instruction) >= 10 and 10 <= len(output) <= 2048

valid_samples = [s for s in dataset if is_valid(s)]
print(f"Valid samples: {len(valid_samples)}")

if len(valid_samples) > SAMPLE_SIZE:
    valid_samples = random.sample(valid_samples, SAMPLE_SIZE)
print(f"Selected samples: {len(valid_samples)}")

In [ ]:
# --- Format into chat template ---
def format_to_chat(sample):
    instruction = clean_text(sample["instruction"])
    context = clean_text(sample.get("input", ""))
    output = clean_text(sample["output"])

    user_msg = instruction
    if context:
        user_msg = f"{instruction}\n\nContext: {context}"

    return {
        "messages": [
            {"role": "system", "content": "You are a helpful financial assistant. Answer financial questions accurately and concisely."},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": output},
        ]
    }

formatted = [format_to_chat(s) for s in valid_samples]

train_data, eval_data = train_test_split(formatted, test_size=EVAL_RATIO, random_state=SEED)
print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

In [ ]:
# --- Save to JSONL ---
Path("data").mkdir(exist_ok=True)

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

save_jsonl(train_data, "data/train.jsonl")
save_jsonl(eval_data, "data/eval.jsonl")
print("Saved data/train.jsonl and data/eval.jsonl")

In [ ]:
# --- Preview a sample ---
print("Sample training example:")
print(json.dumps(train_data[0], indent=2))

## Step 2: QLoRA Fine-Tuning

Load Qwen2.5-1.5B-Instruct in 4-bit, apply LoRA adapters, and train with SFTTrainer.

In [ ]:
# --- Config ---
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "output/financial-qa-qlora"
ADAPTER_PATH = f"{OUTPUT_DIR}/final_adapter"

EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
MAX_SEQ_LEN = 512
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [ ]:
# --- 4-bit Quantization ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Load Tokenizer ---
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load Model ---
print(f"Loading model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("Model loaded!")

In [ ]:
# --- LoRA Config ---
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# --- Prepare datasets ---
train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)
print(f"Train dataset: {len(train_dataset)} | Eval dataset: {len(eval_dataset)}")

In [ ]:
# --- Training Arguments ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=True,
    report_to="none",
    gradient_checkpointing=True,
)

# --- Trainer ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LEN,
)

print("Trainer ready!")

In [ ]:
# --- Train! ---
print("Starting training...")
trainer.train()

In [ ]:
# --- Save LoRA Adapter ---
Path(ADAPTER_PATH).mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to: {ADAPTER_PATH}")

# --- Save training metrics ---
metrics = trainer.state.log_history
with open(f"{OUTPUT_DIR}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Training metrics saved!")

## Step 3: Evaluation - Base vs Fine-Tuned

Compare the base model and fine-tuned model on the eval set using ROUGE scores and qualitative examples.

In [ ]:
# --- Free memory and reload for eval ---
import gc
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared for evaluation.")

In [ ]:
NUM_EVAL_SAMPLES = 50

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

eval_samples = load_jsonl("data/eval.jsonl")[:NUM_EVAL_SAMPLES]
print(f"Evaluating on {len(eval_samples)} samples")

In [ ]:
def generate_response(model, tokenizer, messages, max_new_tokens=256):
    """Generate a response given chat messages."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def compute_rouge(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}
    for pred, ref in zip(predictions, references):
        result = scorer.score(ref, pred)
        for key in scores:
            scores[key].append(result[key].fmeasure)
    return {key: sum(vals) / len(vals) for key, vals in scores.items()}

In [ ]:
# --- Load BASE model ---
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Base model loaded!")

In [ ]:
# --- Base model inference ---
print("Generating base model responses...")
base_predictions = []
references = []
input_prompts = []

for sample in tqdm(eval_samples, desc="Base model"):
    messages = sample["messages"]
    input_msgs = [m for m in messages if m["role"] != "assistant"]
    reference = [m for m in messages if m["role"] == "assistant"][0]["content"]

    pred = generate_response(base_model, tokenizer, input_msgs)
    base_predictions.append(pred)
    references.append(reference)
    input_prompts.append(input_msgs[-1]["content"])

base_rouge = compute_rouge(base_predictions, references)
print(f"\nBase Model ROUGE: {base_rouge}")

In [ ]:
# --- Load Fine-Tuned model (base + LoRA) ---
print("Loading fine-tuned model...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()
print("Fine-tuned model loaded!")

In [ ]:
# --- Fine-tuned model inference ---
print("Generating fine-tuned model responses...")
ft_predictions = []

for sample in tqdm(eval_samples, desc="Fine-tuned model"):
    messages = sample["messages"]
    input_msgs = [m for m in messages if m["role"] != "assistant"]
    pred = generate_response(ft_model, tokenizer, input_msgs)
    ft_predictions.append(pred)

ft_rouge = compute_rouge(ft_predictions, references)
print(f"\nFine-Tuned Model ROUGE: {ft_rouge}")

In [ ]:
# --- Results Comparison ---
print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"{'Metric':<15} {'Base':>10} {'Fine-Tuned':>12} {'Improvement':>13}")
print("-" * 50)
for key in ["rouge1", "rouge2", "rougeL"]:
    diff = ft_rouge[key] - base_rouge[key]
    print(f"{key:<15} {base_rouge[key]:>10.4f} {ft_rouge[key]:>12.4f} {diff:>+13.4f}")
print("=" * 60)

In [ ]:
# --- Save evaluation results ---
Path("evaluation").mkdir(exist_ok=True)

summary = {
    "num_samples": len(eval_samples),
    "base_model": MODEL_NAME,
    "adapter_path": ADAPTER_PATH,
    "base_rouge": base_rouge,
    "finetuned_rouge": ft_rouge,
    "improvement": {key: ft_rouge[key] - base_rouge[key] for key in base_rouge},
}

with open("evaluation/evaluation_results.json", "w") as f:
    json.dump(summary, f, indent=2)

# --- Save qualitative samples ---
qualitative = []
for i in range(min(10, len(eval_samples))):
    qualitative.append({
        "question": input_prompts[i],
        "reference": references[i],
        "base_response": base_predictions[i],
        "finetuned_response": ft_predictions[i],
    })

with open("evaluation/qualitative_samples.json", "w") as f:
    json.dump(qualitative, f, indent=2, ensure_ascii=False)

print("Results saved to evaluation/")

## Step 4: Qualitative Comparison

View side-by-side examples of base vs fine-tuned model responses.

In [ ]:
# --- Display qualitative examples ---
for i, q in enumerate(qualitative[:5]):
    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    print(f"\nQuestion: {q['question'][:200]}")
    print(f"\nReference Answer: {q['reference'][:300]}")
    print(f"\nBase Model: {q['base_response'][:300]}")
    print(f"\nFine-Tuned: {q['finetuned_response'][:300]}")

## Step 5: Save Adapter for Submission

Download the LoRA adapter to upload to HuggingFace or include in your repo.

In [ ]:
# --- Zip adapter for download ---
import shutil
shutil.make_archive("financial-qa-lora-adapter", "zip", ADAPTER_PATH)
print("Adapter zipped as: financial-qa-lora-adapter.zip")
print("Download it from the file browser on the left.")

In [ ]:
# --- (Optional) Push to HuggingFace Hub ---
# Uncomment and set your HF token + repo name to upload

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
# model.push_to_hub("your-username/financial-qa-qwen2.5-1.5b-qlora")
# tokenizer.push_to_hub("your-username/financial-qa-qwen2.5-1.5b-qlora")